# Classical baselines (DDPM + Rectified Flow)

This notebook trains standard classical baselines on **QG channel 1** (64×64, 1k samples) and evaluates them using the same domain-aware metrics used in `notebooks/quantum/qfm_cond_qg.ipynb`.


In [ ]:
%cd /Users/masha/Documents/GSOC/GSoC-Quantum-Diffusion-Model

import numpy as np
import h5py
import torch
from sklearn.model_selection import train_test_split

from utils.baselines import (
    DDPMBaseline,
    RectifiedFlowBaseline,
    DDPMConfig,
    RectifiedFlowConfig,
    TrainConfig,
    SampleConfig,
    get_device,
)
from utils.jet_metrics import compute_jet_metrics, center_by_energy_centroid_zeropad

device = get_device()
print('Device:', device)


In [ ]:
# Data: QG channel 1, 64x64, 1k
QG_channel = 1
filename = f'data/QG{QG_channel}_64x64_1k'

data_X = np.array(h5py.File(filename, 'r')['X'])
print('Raw:', data_X.shape, data_X.dtype)

# Match qfm_cond_qg preprocessing
data_X = data_X.astype(np.float32)
data_X = np.log1p(data_X)
data_X = data_X / (data_X.max() + 1e-8)

x_all = torch.tensor(data_X, dtype=torch.float32).unsqueeze(1)  # (N,1,64,64)
x_all = center_by_energy_centroid_zeropad(x_all)
x_all = x_all.clamp(0.0, 1.0)
print('x_all:', x_all.shape, float(x_all.min()), float(x_all.max()))

# Train/val split
seed = 123
x_train, x_val = train_test_split(x_all, test_size=0.2, random_state=seed, shuffle=True)
print('train/val:', x_train.shape, x_val.shape)


In [ ]:
# Baselines
ddpm = DDPMBaseline(cfg=DDPMConfig(image_size=64, timesteps=200, data_range='01'), device=device)
rf = RectifiedFlowBaseline(
    cfg=RectifiedFlowConfig(image_size=64, data_range='01', solver='rk4', solver_steps=80),
    device=device,
)

train_cfg = TrainConfig(epochs=20, batch_size=64, lr=2e-4, grad_clip=1.0, amp=False)

ddpm_stats = ddpm.fit(x_train, cfg=train_cfg)
rf_stats = rf.fit(x_train, cfg=train_cfg)

print('DDPM:', ddpm_stats)
print('RectifiedFlow:', rf_stats)


In [ ]:
# Sampling
n_eval = 64
samples_ddpm = ddpm.sample(cfg=SampleConfig(n=n_eval, seed=seed))
samples_rf = rf.sample(cfg=SampleConfig(n=n_eval, steps=80, seed=seed))

print('samples_ddpm:', samples_ddpm.shape, float(samples_ddpm.min()), float(samples_ddpm.max()))
print('samples_rf:  ', samples_rf.shape, float(samples_rf.min()), float(samples_rf.max()))


In [ ]:
# Metrics (same core metrics as qfm_cond_qg): E_W1 and RadialL2Log
# Use a fixed real reference subset.
real_ref01 = x_all[:256].to(device)
ddpm01 = samples_ddpm.to(device)
rf01 = samples_rf.to(device)

m_ddpm = compute_jet_metrics(real_ref01, ddpm01, nbins=32, thr_percentile=99.5, centering='roll')
m_rf = compute_jet_metrics(real_ref01, rf01, nbins=32, thr_percentile=99.5, centering='roll')

print('DDPM   E_W1:', m_ddpm['E_w1'], 'RadialL2Log:', m_ddpm['radial_l2_log'])
print('RF     E_W1:', m_rf['E_w1'], 'RadialL2Log:', m_rf['radial_l2_log'])

# (Optional) additional jet metrics returned by compute_jet_metrics
print('DDPM extra:', {k: v for k, v in m_ddpm.items() if k not in ['E_w1', 'radial_l2_log']})
